# Lecture 01: Introduction — JAX, Autodiff, and Physics-vs-ML Modeling

**Machine Learning Applications in Physics (PHYG004)** — Sogang University, 2026 Spring

> **Data type for this lecture:** Toy/Synthetic (analytic potentials, Ising MC)
> Each lecture will carry such a label so you always know whether you are working
> with real measured/computed data or an idealized toy system.

---

## Overview

This notebook covers the practical tools you will use all semester, introduced
immediately through physics examples:

1. JAX array operations and the NumPy → JAX migration (brief reference)
2. Explicit PRNG keys — JAX's functional random-number system
3. `jax.grad` — exact derivatives from Python code (force from potential)
4. `jax.vmap` — vectorised computation over batches (harmonic oscillator ensemble)
5. `jax.jit` — XLA compilation for speed
6. **Benchmark System: 2D Ising Model** — our first recurring spine system
7. Named Spine declaration: the two benchmark systems that will follow us all semester

## Learning Objectives

- Understand why JAX uses immutable arrays and explicit PRNG keys
- Compute forces/curvatures with `jax.grad` without writing any derivative by hand
- Use `vmap` to vectorise over physical parameters
- Run a compiled Metropolis MC simulation with `jit` + `lax.fori_loop`
- Know the two *named spine* systems that will reappear throughout the course


In [ ]:
# Install dependencies (Google Colab — skip if running locally with .venv)
!pip install -q jax jaxlib optax flax matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility seed — used throughout the notebook
key = random.PRNGKey(42)

print(f"JAX version : {jax.__version__}")
print(f"Devices     : {jax.devices()}")
print(f"Backend     : {jax.default_backend()}")
print()
print("Tip: on Colab with GPU enabled you will see a 'cuda' device above.")
print("     Runtime → Change runtime type → GPU")

---
## § 1 — Reference: NumPy ↔ JAX One-Slide Summary

If you already know NumPy, you already know most of JAX.
The table below shows the key differences; the only one that is *surprising* is
the **immutability** of JAX arrays.

| NumPy | JAX equivalent | Note |
|---|---|---|
| `np.array([1,2,3])` | `jnp.array([1,2,3])` | same syntax |
| `a[2] = 99` | `a = a.at[2].set(99)` | JAX creates a *new* array |
| `np.random.randn(5)` | `random.normal(key, (5,))` | explicit PRNG key required |
| `np.dot(a, b)` | `jnp.dot(a, b)` or `a @ b` | identical |
| `import numpy as np` | `import jax.numpy as jnp` | drop-in for most ops |

### Array immutability

```python
# NumPy: in-place works
arr = np.array([1, 2, 3])
arr[1] = 99                   # OK

# JAX: raises TypeError
arr = jnp.array([1, 2, 3])
arr[1] = 99                   # TypeError!
arr = arr.at[1].set(99)       # correct JAX idiom
```

### Matmul benchmark (pre-computed result)

| Hardware | NumPy (2000×2000) | JAX CPU | JAX GPU (T4) |
|---|---|---|---|
| Google Colab CPU | ~0.07 s | ~0.05 s | — |
| Google Colab GPU | — | — | ~0.003 s |

Run the benchmark yourself if you are curious, but we skip it here to keep
the session focused on physics.


In [ ]:
# Brief demonstration of immutability — the one gotcha from NumPy

jax_arr = jnp.array([1, 2, 3, 4, 5])

try:
    jax_arr[2] = 99          # will raise TypeError
except TypeError as e:
    print("Expected error:", e)

# Correct JAX idiom: .at[].set() returns a NEW array
jax_arr_new = jax_arr.at[2].set(99)
print(f"Original : {jax_arr}")
print(f"Updated  : {jax_arr_new}")
print(f"Shapes   : {jax_arr.shape}, {jax_arr_new.shape}")

---
## § 2 — Explicit PRNG Keys

JAX replaces the hidden global random state of NumPy with **explicit, functional keys**.

Why does this matter for physics?

- **Reproducibility**: the same key always yields the same realisation.
- **Parallelism**: independent sub-keys can be sent to different devices.
- **Functional purity**: no hidden side effects — functions are predictable.

### Key rule: never reuse a key

Split it with `random.split` whenever you need a new stream:

```python
key, subkey = random.split(key)       # split into 2
key, sk1, sk2 = random.split(key, 3)  # split into 3
```


In [ ]:
# PRNG key demonstration

key = random.PRNGKey(42)

# Same key → same samples (reproducible)
x1 = random.normal(key, shape=(4,))
x2 = random.normal(key, shape=(4,))
print(f"x1 (key 42): {x1}")
print(f"x2 (key 42): {x2}")
print(f"x1 == x2 : {jnp.allclose(x1, x2)}")  # True

# Split key → independent streams
key, subkey_a, subkey_b = random.split(key, 3)
ya = random.normal(subkey_a, shape=(4,))
yb = random.normal(subkey_b, shape=(4,))
print(f"\nya (split a): {ya}")
print(f"yb (split b): {yb}")
print(f"ya == yb    : {jnp.allclose(ya, yb)}")  # False

---
## § 3 — Automatic Differentiation with `jax.grad`

In physics we constantly need derivatives:
- **Force** $F = -\nabla V$ from a potential $V$
- **Equations of motion** from a Lagrangian $\mathcal{L}$
- **Response functions**, susceptibilities, …

JAX computes **exact** derivatives via automatic differentiation (AD), not
finite differences. The result is exact up to floating-point precision and
does *not* depend on a step size $h$.

```python
df  = jax.grad(f)        # first derivative  df/dx
d2f = jax.grad(jax.grad(f))  # second derivative d²f/dx²
```

`jax.grad` differentiates through any JAX-compatible Python — loops, branches,
nested functions — as long as the computation is purely functional.


In [ ]:
# Physics example 1: Harmonic oscillator  V(x) = ½ k x²
# Force: F = -dV/dx  →  expected -kx

k = 2.0

def V_ho(x):
    """Harmonic oscillator potential."""
    return 0.5 * k * x**2

# Force = negative gradient of potential
F_ho = jax.grad(lambda x: -V_ho(x))

# Curvature = second derivative (spring constant recoverable from curvature!)
curvature = jax.grad(jax.grad(V_ho))

print("Harmonic oscillator  V(x) = ½kx²,  k =", k)
print(f"{'x':>6}  {'V(x)':>8}  {'F(x)':>8}  {'F expected':>12}  {'d²V/dx²':>10}")
print("-" * 55)
for xv in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    xf = float(xv)
    print(f"{xv:>6.1f}  {V_ho(xf):>8.3f}  {F_ho(xf):>8.3f}  {-k*xf:>12.3f}  {curvature(xf):>10.3f}")

print()
print(f"Curvature = d²V/dx² = k = {curvature(0.0):.1f}  ✓")

### Physics example 2: Lennard-Jones potential

$$V_{LJ}(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

The force is $F = -dV/dr$.
The equilibrium distance is $r_{\rm eq} = 2^{1/6}\sigma$ where $F=0$.

We can verify this **checksum** using `jax.grad`.


In [ ]:
# Lennard-Jones force via jax.grad

eps = 1.0  # energy scale
sig = 1.0  # length scale

def V_lj(r):
    """Lennard-Jones potential."""
    sr6  = (sig / r)**6
    return 4.0 * eps * (sr6**2 - sr6)

F_lj = jax.grad(lambda r: -V_lj(r))  # force = -dV/dr

r_vals = jnp.linspace(0.9, 3.0, 200)

# Vectorise over r with vmap
V_vals = jax.vmap(V_lj)(r_vals)
F_vals = jax.vmap(F_lj)(r_vals)

# Checksum: force should be 0 at r_eq = 2^(1/6) sigma
r_eq = 2.0**(1.0/6.0) * sig
F_eq = F_lj(r_eq)
print(f"Equilibrium distance r_eq = 2^(1/6) σ = {r_eq:.4f}")
print(f"Force at r_eq             = {F_eq:.2e}  (should be ≈ 0)  ✓")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(r_vals, V_vals, 'b-', lw=2)
ax1.axhline(0, color='k', lw=0.5)
ax1.axvline(r_eq, color='r', ls='--', alpha=0.7, label=f'$r_{{eq}}={r_eq:.2f}$')
ax1.set_ylim(-2, 2)
ax1.set_xlabel('r')
ax1.set_ylabel('V(r)')
ax1.set_title('Lennard-Jones Potential')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(r_vals, F_vals, 'r-', lw=2)
ax2.axhline(0, color='k', lw=0.5)
ax2.axvline(r_eq, color='r', ls='--', alpha=0.7, label=f'$r_{{eq}}={r_eq:.2f}$')
ax2.set_ylim(-5, 10)
ax2.set_xlabel('r')
ax2.set_ylabel('F(r)')
ax2.set_title('LJ Force (via jax.grad)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Lennard-Jones via jax.grad — force computed automatically', fontsize=12)
plt.tight_layout()
plt.show()
print("Both curves computed without writing a single derivative formula!")

---
## § 4 — Vectorised Computation with `jax.vmap`

`vmap` transforms a function that acts on a *single* example into one that acts
on a *batch* — without any explicit for-loop.

**Physics analogy**: computing the same observable for an ensemble of systems
differing only in one parameter (temperature, spring constant, …).

```python
# Without vmap: explicit loop
results = [f(x_i) for x_i in batch]

# With vmap: one call, fully parallelisable
results = jax.vmap(f)(batch)
```


In [ ]:
# vmap example: curvature of harmonic oscillator for multiple spring constants

k_vals = jnp.array([0.5, 1.0, 2.0, 4.0])  # four different spring constants

def curvature_ho(k, x=0.0):
    """Second derivative d²V/dx² of ½kx² evaluated at x."""
    def _V(x_):
        return 0.5 * k * x_**2
    return jax.grad(jax.grad(_V))(x)

# Vectorise over k
curvatures = jax.vmap(curvature_ho)(k_vals)

print("Harmonic oscillator curvature = d²V/dx² = k (checksum)")
print(f"{'k':>6}  {'d²V/dx²':>10}  {'match':>8}")
print("-" * 30)
for ki, ci in zip(k_vals, curvatures):
    match = "✓" if abs(float(ci) - float(ki)) < 1e-5 else "✗"
    print(f"{ki:>6.2f}  {ci:>10.4f}  {match:>8}")

print()
print("k_vals   :", k_vals)
print("curvature:", curvatures)
print("Shapes — k_vals:", k_vals.shape, " curvatures:", curvatures.shape)

---
## § 5 — JIT Compilation with `jax.jit`

`jax.jit` traces your function and compiles it with **XLA** (Accelerated Linear
Algebra). After the one-time compilation cost, subsequent calls are much faster.

**Rule of thumb**: JIT-compile the inner loop of any simulation.

```python
@jax.jit
def sweep(state, key, beta): ...   # compiled once, fast every call after
```

The key constraint: the function must be *functionally pure* — no Python
side-effects inside the compiled region. Loops inside JIT must use
`jax.lax.fori_loop` (or `scan`) instead of Python `for`.


In [ ]:
import time

# A simple function with many ops — useful for JIT demo
def matpow(x):
    for _ in range(100):
        x = x @ x.T / (jnp.linalg.norm(x) + 1e-8)
    return x

fast_matpow = jax.jit(matpow)

key_demo, sk = random.split(random.PRNGKey(0))
x_demo = random.normal(sk, (50, 50))

# Warm up (first call includes compilation)
_ = fast_matpow(x_demo).block_until_ready()

# Time without JIT
t0 = time.time()
for _ in range(5):
    _ = matpow(x_demo).block_until_ready()
t_nojit = (time.time() - t0) / 5

# Time with JIT
t0 = time.time()
for _ in range(5):
    _ = fast_matpow(x_demo).block_until_ready()
t_jit = (time.time() - t0) / 5

print(f"Without JIT : {t_nojit*1000:.2f} ms")
print(f"With JIT    : {t_jit*1000:.2f} ms")
print(f"Speedup     : {t_nojit/t_jit:.1f}×")
print("(GPU Colab speedup will be much larger — try it!)")

---
## § 6 — Benchmark System: 2D Ising Model

$$H = -J \sum_{\langle i,j \rangle} s_i s_j, \qquad s_i \in \{-1, +1\}$$

The 2D Ising model on a square lattice is exactly solvable (Onsager 1944):

$$T_c = \frac{2J}{\ln(1+\sqrt{2})} \approx 2.269\; J/k_B$$

Below $T_c$ the system orders (all spins aligned); above $T_c$ thermal
fluctuations disorder it.

**Why here?** This is the first of our two *named spine* systems. The Ising
model will reappear in later lectures as:
- **L06/L07**: Convolutional network phase classifier (translate spin images → phase label)
- **L14**: VAE latent space discovers magnetisation as an order parameter

We keep the simulation minimal here: one lattice size, one temperature sweep.
We will extend it when we revisit it.

### JAX features exercised

| Feature | Where used |
|---|---|
| `jax.jit` | compiles the Metropolis sweep |
| `lax.fori_loop` | JIT-compatible inner loop (L² flip attempts) |
| `.at[].set()` | immutable spin-flip update |
| Explicit PRNG | key splitting inside the compiled loop |


In [ ]:
# ── Ising model helpers ──────────────────────────────────────────────────────

def init_lattice(key, L):
    """Random ±1 spin lattice of size L×L."""
    return 2 * random.bernoulli(key, shape=(L, L)).astype(jnp.int8) - 1

def compute_energy(spins):
    """Total energy H = -J Σ s_i s_j (J=1, periodic BC via jnp.roll)."""
    return -jnp.sum(
        spins * (jnp.roll(spins, 1, axis=0) + jnp.roll(spins, 1, axis=1))
    ).astype(jnp.float32)

def compute_magnetization(spins):
    """Magnetization per spin m = (1/N) Σ s_i."""
    return jnp.mean(spins.astype(jnp.float32))

# Quick sanity check
key = random.PRNGKey(42)
L = 16   # small lattice for quick demo
spins0 = init_lattice(key, L)
print(f"Lattice  : {L}×{L}  ({L*L} spins)")
print(f"E        : {compute_energy(spins0):.0f}")
print(f"|m|      : {jnp.abs(compute_magnetization(spins0)):.4f}")
print(f"spins shape: {spins0.shape}, dtype: {spins0.dtype}")

In [ ]:
# ── Metropolis sweep (JIT + lax.fori_loop) ────────────────────────────────────

@jax.jit
def metropolis_sweep(spins, key, beta):
    """One Metropolis sweep = L² single-spin-flip attempts."""
    L = spins.shape[0]
    N = L * L

    def step(i, carry):
        spins, key = carry
        key, k1, k2, k3 = random.split(key, 4)

        # Random site
        x = random.randint(k1, (), 0, L)
        y = random.randint(k2, (), 0, L)

        # Energy change: ΔE = 2 s_xy Σ_nn s_nn
        s = spins[x, y]
        nn_sum = (
            spins[(x + 1) % L, y]
            + spins[(x - 1) % L, y]
            + spins[x, (y + 1) % L]
            + spins[x, (y - 1) % L]
        )
        dE = (2.0 * s * nn_sum).astype(jnp.float32)

        # Metropolis acceptance
        accept = (dE <= 0) | (random.uniform(k3) < jnp.exp(-beta * dE))
        new_s = jnp.where(accept, -s, s)
        spins = spins.at[x, y].set(new_s)
        return (spins, key)

    spins, key = lax.fori_loop(0, N, step, (spins, key))
    return spins, key

print("metropolis_sweep defined (JIT-compiled).")
print("First call will trigger compilation; subsequent calls use cached XLA.")

# Warm up compilation
key, sk = random.split(random.PRNGKey(0))
spins_demo = init_lattice(sk, 16)
key, sk2 = random.split(key)
spins_demo, _ = metropolis_sweep(spins_demo, sk2, beta=1.0)
print(f"After 1 sweep — |m| = {jnp.abs(compute_magnetization(spins_demo)):.4f}")

In [ ]:
# ── Temperature sweep ─────────────────────────────────────────────────────────

def run_ising(key, L, T, n_warmup=150, n_measure=200):
    """Run MC at temperature T, return mean |m| and final spin config."""
    beta = 1.0 / T
    key, init_key = random.split(key)
    spins = init_lattice(init_key, L)

    # Thermalisation (discard)
    for _ in range(n_warmup):
        spins, key = metropolis_sweep(spins, key, beta)

    # Measurement
    mag_acc = 0.0
    for _ in range(n_measure):
        spins, key = metropolis_sweep(spins, key, beta)
        mag_acc += jnp.abs(compute_magnetization(spins))

    return mag_acc / n_measure, spins

# ─ Run
L = 16                           # L=16 is fast; try L=32 for a sharper transition
T_c = 2.0 / jnp.log(1 + jnp.sqrt(2.0))
temps = jnp.linspace(1.2, 3.5, 12)
key = random.PRNGKey(7)

print(f"Running Ising MC   L={L}   T_c(exact)={float(T_c):.3f}")
print(f"{'T':>6}  {'<|m|>':>8}  {'phase':>12}")
print("-" * 30)

mags = []
snapshots = {}
for idx, T in enumerate(temps):
    key, sk = random.split(key)
    m, final_spins = run_ising(sk, L, float(T), n_warmup=150, n_measure=200)
    mags.append(float(m))
    phase = "ordered" if float(T) < float(T_c) - 0.3 else (
            "critical" if abs(float(T) - float(T_c)) < 0.5 else "disordered")
    print(f"{T:>6.2f}  {m:>8.4f}  {phase:>12}")
    if idx in [0, len(temps) // 2, len(temps) - 1]:
        snapshots[float(T)] = final_spins

print(f"\nDone — {len(temps)} temperature points.")

In [ ]:
# ── Plot: magnetization + spin snapshots ──────────────────────────────────────

fig = plt.figure(figsize=(14, 4))

# Left: magnetization curve
ax0 = fig.add_subplot(1, 4, 1)
ax0.plot(temps, mags, 'o-', color='royalblue', ms=5, label=f'MC (L={L})')
ax0.axvline(float(T_c), color='red', ls='--', alpha=0.7,
            label=f'$T_c$ = {float(T_c):.3f}')
ax0.set_xlabel('Temperature $T$ [$J/k_B$]')
ax0.set_ylabel(r'$\langle |m| \rangle$')
ax0.set_title('Spontaneous Magnetization')
ax0.set_ylim(-0.05, 1.05)
ax0.legend(fontsize=8)
ax0.grid(True, alpha=0.3)

# Right: three spin snapshots
labels = ['ordered\n(T ≪ T_c)', 'near T_c', 'disordered\n(T ≫ T_c)']
for k, (T_val, spins) in enumerate(sorted(snapshots.items())):
    ax = fig.add_subplot(1, 4, k + 2)
    ax.imshow(spins, cmap='coolwarm', vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(f'T = {T_val:.2f}\n{labels[k]}', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('2D Ising Model (L=16) — first encounter with our spine system', fontsize=12)
plt.tight_layout()
plt.show()

---
## Exercises

Work through these before the next session. Each has a **checksum** so you can
verify your answer.

---

### Exercise 1 — `jax.grad` and Lennard-Jones force

Compute the force $F(r) = -dV/dr$ of the Lennard-Jones potential

$$V_{LJ}(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right]$$

using `jax.grad`, and compare with a NumPy finite-difference approximation
$F_{\rm FD}(r) \approx -[V(r+h) - V(r-h)] / (2h)$ for $h = 10^{-4}$.

**Checksum**: at the equilibrium distance $r_{\rm eq} = 2^{1/6}\sigma$ (set $\sigma=\varepsilon=1$)
the force should satisfy $|F(r_{\rm eq})| < 10^{-5}$.

---

### Exercise 2 — PRNG pattern

Use `random.split` to initialise **three** independent $L=8$ Ising lattices
(`init_lattice` from § 6) each with a different sub-key, and print the initial
magnetisation of each.

Then demonstrate that reusing the **same** key produces identical lattices.

---

### Checkpoint — `vmap` over spring constants

Use `jax.vmap` to compute the curvature $d^2V/dx^2$ of the harmonic oscillator
$V(x) = \frac{1}{2}kx^2$ at $x = 0$ for $k \in [0.5, 1.0, 2.0, 4.0]$ in one call.

**Checksum**: curvature equals $k$ for all entries.

---

### Checkpoint — Named Spine recognition

Read the Named Spine declaration in § 7 below, then answer briefly:

(a) What physical quantity does the Ising model compute in this notebook?

(b) In which later lecture does the Ising model's spin configuration become the
    *input* to a convolutional neural network?

(c) In which lecture does a VAE latent dimension track the magnetisation?


---
## § 7 — Named Spine: Benchmark Systems for the Semester

> **This box is important.** Two physical systems will reappear throughout the
> course as canonical test cases for every new method we introduce. Meeting them
> now in their simplest form lets you focus on the *method* when they reappear,
> rather than on learning the physics.

---

### Spine 1 — 2D Ising Model  *(discrete spins, phase transition)*

$$H = -J\sum_{\langle i,j\rangle} s_i s_j, \quad s_i \in \{-1,+1\}$$

| Lecture | What we do with the Ising model |
|---|---|
| **L01** (here) | First encounter: Metropolis MC, spontaneous magnetisation |
| **L06** | LeNet architecture trained to classify spin images → ordered/disordered phase |
| **L07** | Real-data CNN hands-on; Ising from-scratch vs transfer-learned classifier |
| **L14** | VAE encodes Ising spin configurations; latent axis ≈ magnetisation $m$ — *discovering an order parameter* |

The Ising model is the minimal example of a **phase transition**, a concept
central to statistical physics. Recognising it in a latent space (L14) is the
payoff that closes the loop.

---

### Spine 2 — Boltzmann Double-Well  *(continuous coordinate, bimodal target)*

$$V(x) = (x^2 - 1)^2, \qquad p(x) \propto e^{-\beta V(x)}$$

At low temperature the probability mass splits between two minima, making it
the minimal **multimodal** target — the stress test for every generative model.

| Lecture | What we do with the double-well |
|---|---|
| **L02** | Motivation lecture: gradient descent in a 1D/2D potential landscape |
| **L15** | Normalizing flow (RealNVP) trained to sample from $p(x)$ |
| **L16** | Flow matching: velocity field transports Gaussian noise → double-well samples |
| **L17** | Diffusion model (DDPM): reverse SDE recovers double-well distribution |

By L17 you will have applied four different generative methods to the *same*
target and can directly compare their behaviour near the barrier.

---

> **Connection to course philosophy** (CLAUDE.md)
>
> *Loss function ~ Free energy; latent space ~ order parameter;
>  equivariance ~ symmetry; sampling ~ Boltzmann distribution.*
>
> The Ising model makes the first three tangible; the double-well makes the
> last one tangible. Together they form the physical thread that runs through
> every block of the course.


---
## Summary

| JAX feature | What we demonstrated |
|---|---|
| `jnp` arrays | immutable; `.at[].set()` idiom |
| `random.PRNGKey` / `random.split` | reproducible, independent random streams |
| `jax.grad` | exact force from potential (harmonic, Lennard-Jones) |
| `jax.vmap` | curvature over ensemble of spring constants |
| `jax.jit` + `lax.fori_loop` | compiled Metropolis sweep |
| Named Spine | Ising (discrete phase transition) + double-well (multimodal Boltzmann) |

### Resources

- JAX documentation: [docs.jax.dev](https://docs.jax.dev)
- JAX quickstart: [docs.jax.dev/en/latest/quickstart.html](https://docs.jax.dev/en/latest/quickstart.html)
- Onsager solution: Onsager, L. (1944). *Crystal Statistics I.* Phys. Rev. 65, 117.
